# Lab 0 · เรียก LLM ครั้งแรก

Notebook นี้ใช้เวลาราว 15–20 นาที เป้าหมายคือยืนยันว่าเครื่องของคุณเรียก LLM ผ่าน LangChain ได้ และได้เห็นแนวคิดจาก Module 0 ทำงานจริง: message, streaming, temperature, token และ structured output

**ก่อนรัน:** เลือก kernel เป็น `.venv` ของโฟลเดอร์ `agentic-ai` (มุมขวาบนของ notebook) แล้วรันทีละ cell ด้วย `Shift+Enter`

## 1. เลือก model

`get_llm()` อ่านค่าจากไฟล์ `.env` ว่าจะใช้ provider และ model ตัวไหน ทุก lab ในวิชานี้ใช้ function เดียวกันนี้ คุณจึงสลับระหว่าง local model กับ API ได้โดยไม่ต้องแก้ code

In [ ]:
from agentlab import get_llm, provider_settings

provider, model, _ = provider_settings()
print(f"provider = {provider}\nmodel    = {model}")

llm = get_llm()

## 2. เรียกครั้งแรก

`invoke()` ส่งข้อความไปแล้วรอคำตอบทั้งก้อน สิ่งที่ได้กลับมาคือ message ที่มีทั้งข้อความและจำนวน token ที่ใช้

In [ ]:
reply = llm.invoke("อธิบายคำว่า token ในบริบทของ LLM ภายใน 2 ประโยค")

print(reply.text)
print("\nusage:", reply.usage_metadata)

## 3. Message และ role

ใน Module 0 เราเห็นว่า API รับรายการ message ที่มี role กำกับ `system` กำหนดกติกา ส่วน `user` คือคำถาม ลองแก้ข้อความใน `system` แล้วดูว่าคำตอบเปลี่ยนอย่างไร

In [ ]:
messages = [
    ("system", "You are an accounting assistant. Answer in Thai, in one short sentence."),
    ("user", "ใบกำกับภาษีต่างจากใบเสร็จรับเงินอย่างไร"),
]
print(llm.invoke(messages).text)

## 4. Streaming

Model สร้างคำตอบทีละ token `stream()` ให้เราเห็นแต่ละชิ้นทันทีที่สร้างเสร็จ แทนที่จะรอจนจบ

In [ ]:
for chunk in llm.stream("นับเลข 1 ถึง 10 เป็นภาษาไทย คั่นด้วยจุลภาค"):
    print(chunk.text, end="|", flush=True)

## 5. Temperature

รัน prompt เดิม 3 ครั้งที่ temperature 0 และอีก 3 ครั้งที่ 1.2 แล้วเทียบว่าชุดไหนคำตอบซ้ำกันมากกว่า

In [ ]:
prompt = "ตั้งชื่อร้านกาแฟ 1 ชื่อ ตอบเฉพาะชื่อร้าน"

for temperature in (0, 1.2):
    model_t = get_llm(temperature=temperature)
    names = [model_t.invoke(prompt).text.strip() for _ in range(3)]
    print(f"temperature={temperature}: {names}")

## 6. ภาษาไทยใช้ token มากกว่าจริงหรือไม่

สองประโยคด้านล่างมีความหมายเดียวกัน เราดูจำนวน input token ที่ model รายงานกลับมา ตัวเลขรวม token ของโครง message ไว้ด้วย ซึ่งเท่ากันทั้งสองกรณี ส่วนต่างจึงมาจากตัวประโยค

In [ ]:
sentences = {
    "en": "The customer must pay the invoice within thirty days of the delivery date.",
    "th": "ลูกค้าต้องชำระเงินตามใบแจ้งหนี้ภายในสามสิบวันนับจากวันที่ส่งมอบสินค้า",
}
for lang, sentence in sentences.items():
    usage = llm.invoke(sentence + "\n\nReply with: ok").usage_metadata
    print(f"{lang}: {len(sentence):3d} characters -> {usage['input_tokens']} input tokens")

## 7. Structured output

งาน document intelligence ต้องการข้อมูลที่โปรแกรมนำไปใช้ต่อได้ เรากำหนด schema ด้วย Pydantic แล้วให้ model ตอบตามนั้น นี่คือเทคนิคหลักของ Lab 1

In [ ]:
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    invoice_no: str = Field(description="Invoice number exactly as printed")
    seller: str
    total: float = Field(description="Grand total including VAT")
    currency: str = Field(description="ISO currency code such as THB")


text = """
ใบแจ้งหนี้ เลขที่ INV-2026-0042
ผู้ขาย: บริษัท สยามเปเปอร์ จำกัด
กระดาษ A4 80 แกรม 50 รีม @ 110.00 = 5,500.00
ภาษีมูลค่าเพิ่ม 7% = 385.00
รวมทั้งสิ้น 5,885.00 บาท
"""

invoice = llm.with_structured_output(Invoice).invoke(f"Extract the invoice fields.\n{text}")
print(invoice)
print("total is a number:", invoice.total + 1)

## 8. สลับ provider

ส่งชื่อ provider ให้ `get_llm()` เพื่อใช้ตัวอื่นชั่วคราว cell นี้ลองทุก provider ที่คุณตั้งค่าไว้ ตัวที่ยังไม่มี key จะถูกข้าม

In [ ]:
import time

from agentlab import PROVIDERS

for name in PROVIDERS:
    try:
        start = time.time()
        answer = get_llm(name).invoke("Reply with exactly one word: ready").text.strip()
        print(f"{name:10s} {answer!r} in {time.time() - start:.1f}s")
    except Exception as error:
        print(f"{name:10s} skipped: {str(error)[:90]}")

## ลองทำเอง

1. แก้ `system` message ในข้อ 3 ให้ model ตอบเป็นภาษาอังกฤษแบบ bullet 3 ข้อ
2. เพิ่ม field `vat: float` ใน `Invoice` แล้วรันข้อ 7 ใหม่ model ดึงค่าได้ถูกหรือไม่
3. แก้ยอดรวมในข้อความ invoice ให้ไม่ตรงกับผลบวกของรายการ แล้วถาม model ว่ายอดรวมถูกหรือไม่ จดไว้ว่า model ตอบอย่างไร เราจะกลับมาที่ปัญหานี้ใน Lab 2

**สิ่งที่ต้องส่ง:** ดูขั้นตอนสุดท้ายในคู่มือ Lab 0